In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO
from PIL import Image
import shutil
import pandas as pd
from source import image_id_converter as img_idc
from source.db_loader import MLDataLoader
from source.db_loader import delete_images
#from source import sort_img_files as sif
from source import llm_input as llm_i
from source import llm_output as llm_o
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

In [ ]:
import ollama
import json
import re
import pickle

In [ ]:
import pandas as pd
import psycopg2
from dotenv import load_dotenv


In [ ]:
import glob


In [ ]:
from psycopg2.extras import execute_batch
from typing import Dict, List, Optional

In [ ]:
# Standard library imports
import sys
from datetime import datetime
from pathlib import Path

# Database connection library
# psycopg2: PostgreSQL adapter for Python - handles all DB communication
import psycopg2
from psycopg2 import extras  # extras provides advanced features like Json adapter

# Data manipulation
import pandas as pd  # For handling CSV files and DataFrames

# Environment variable management
# python-dotenv: Loads database credentials from .env file (keeps passwords out of code)
from dotenv import load_dotenv

In [ ]:
def get_existing_image_ids(source_filter, db_env, conn=None, cur=None):
    """
    Get all source_image_ids from database.
    
    Parameters:
    -----------
    source_filter : str
        Source to filter by (e.g., 'giub')
    conn : connection object, optional
        Existing database connection (if None, creates new one)
    cur : cursor object, optional
        Existing cursor (if None, creates new one)
    
    Returns:
    --------
    list : List of source_image_ids
    """
    #import psycopg2
    #from dotenv import load_dotenv
    #import os
    
    close_after = False
    if conn is None:
        load_dotenv()
        conn = psycopg2.connect(
            dbname=os.getenv('DB_NAME'),
            user=os.getenv('DB_USER'),
            password=os.getenv('DB_PASSWORD'),
            host=os.getenv('DB_HOST'),
            port=os.getenv('DB_PORT')
        )
        cur = conn.cursor()
        close_after = True
    elif cur is None:
        cur = conn.cursor()
    
    # Added WHERE clause with parameter
    cur.execute("""
        SELECT DISTINCT source_image_id 
        FROM images 
        WHERE source = %s 
        ORDER BY source_image_id;
    """, (source_filter,))
                    
    existing_ids = [row[0] for row in cur.fetchall()]
    #print(f"Existing image IDs for source '{source_filter}': {existing_ids}")
    
    if close_after:
        cur.close()
        conn.close()
    
    return existing_ids

### Set database_name: 

In [ ]:
#database_name = 'image_analysis_dev'
database_name = 'image_analysis_test'

#db_env = '.env'
db_env = '.env.test'

# Load test environment (override=True ensures it replaces any previously loaded values)
load_dotenv(db_env, override=True)

# Explicit check — you see exactly what you're connecting to
db_name = os.getenv('DB_NAME')
print(f"Connecting to: {db_name}")
assert db_name == database_name, f"Wrong database loaded: {db_name}"


print("✅ Libraries loaded")


### Empty database:

#### If necessary use this:
loader.conn.rollback()

### Set paths:

In [ ]:
project_path = Path.cwd()

# visual_genome_path = (project_path/ '..' /'data_folders' / 'visual_genome_data').resolve()
# visual_genome_proc_path = (project_path/ '..' /'data_folders' / 'visual_genome_proc_data').resolve()

# data_path_comb = (project_path/ '..' /'data_folders'/'combined_data').resolve()

data_path = (project_path/ '..' /'data_folders'/'data').resolve()
image_dir = (project_path/ '..' /'data_folders'/'data_1').resolve()
#data_path_comb = (project_path/ '..' /'data_folders'/'combined_data').resolve()


### Check files (most recent on top):

In [ ]:
# sorted(os.listdir(data_path_comb), key=lambda f: os.path.getmtime(os.path.join(data_path_comb, f)), reverse=True)
sorted(os.listdir(data_path), key=lambda f: os.path.getmtime(os.path.join(data_path, f)), reverse=True)


### Set timestamp id:

In [ ]:
timestamp_id = '20260325_201946'

### Set file paramters: 

In [ ]:
#file_source_1 = 'visual_genome' 
#file_source_2 = 'swiss_topo'
#file_source = 'giub'
#file_extension = '.jpg' 
#filename_tag = 'visual_genome_proc' # First batch of images obtained from the online visual genome data set)
#labels_file = 'labels.csv' # File containing ids and labels.
labels_file = 'labels_new.csv'
# times_file = 'times_clustering_pipeline_20260312_120235.pkl' # File containing timestamp and duration of analysis-run.
# times_file = 'times_clustering_pipeline_20260314_211803.pkl'
# times_file = 'times_clustering_20260323_174520.pkl'
# times_file = 'times_clustering_20260324_223646.pkl'
times_file = 'times_clustering_pipeline_20260325_201946.pkl'
# times_file = 'times_clustering_pipeline_20260325_204829.pkl'
#metadata_results_file = 'results_clustering_pipeline_20260312_120235.pkl'
#metadata_results_file = 'results_clustering_pipeline_20260314_211803.pkl'
#metadata_results_file = 'results_clustering_20260323_174520.pkl'
#metadata_results_file = 'results_clustering_20260324_223646.pkl'
metadata_results_file = 'results_clustering_pipeline_20260325_201946.pkl'
# metadata_results_file = 'results_clustering_pipeline_20260325_204829.pkl'
#responses_file = 'responses_llm_people_detect_multi_approach_20260213_144241.pkl'
#model_name_file = 'minicpm_v_model_info.txt'
#metadata_results_file = 'metadata_results_clustering.pkl'




### Load labels file: 

In [ ]:
# file_path = data_path_comb / labels_file
file_path = data_path / labels_file
label_data = pd.read_csv(file_path)
label_data.head()

In [ ]:
label_data.file_paths[0]

### Add original file paths to labels data:

#### Necessary for image files where the original format is different from the format that was used for analysis:


In [ ]:

# Get all image files in image_dir
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith('.tif')]
print(f"\nFound {len(image_files)} jpg files in image directory")

In [ ]:
# Build a dict mapping trailing integer -> full file path
id_to_filepath = {}
for filename in image_files:
    match = re.search(r'(\d+)(?=\.\w+$)', filename)
    if match:
        image_id = int(match.group(1))
        id_to_filepath[image_id] = str(image_dir / filename)
    else:
        print(f"Warning: could not extract integer from filename: {filename}")

print(f"Successfully mapped {len(id_to_filepath)} filenames to image_ids")

In [ ]:

# Match image_ids in labels_df to file paths
label_data['file_paths'] = label_data['image_id'].map(id_to_filepath)
#label_data['file_source'] = file_source

# Check for unmatched ids
n_missing = label_data['file_paths'].isna().sum()
if n_missing > 0:
    print(f"\nWarning: {n_missing} image_ids could not be matched to a file")
    print(label_data[label_data['file_paths'].isna()])
else:
    print(f"\nAll image_ids successfully matched to file paths")

print(label_data.head())

In [ ]:
label_data.file_paths[0]

### Set label name: 

In [ ]:
label_name = 'is_map'

In [ ]:
filepaths = list(label_data.file_paths)
filepaths[0]

In [ ]:
filepaths[-1]

### Label data by file source:

In [ ]:
file_sources = list(set(label_data.file_source))
file_sources

In [ ]:
label_data_from_sources = {}
for file_source in file_sources:
    label_data_from_sources[file_source] = label_data[label_data.file_source == file_source]
print(label_data_from_sources.keys())

    

In [ ]:
label_data_from_sources[file_source].shape

In [ ]:
# label_data_from_sources[file_source_2].shape

### Set analysis run parameters: 

In [ ]:
# analysis_type = 'clustering with autoencoder'
# model_name = 'cdae_model_20260309_111030'
# model_name = 'vae_model_20260314_211803'
model_name = timestamp_id + '/var_conv_ae_20260325_201946'
#model_name = 'var_conv_ae_20260325_204829'
# #model_version = model_version
# python_script = 'img_to_pytorch.ipynb'
model_path = data_path / model_name
os.listdir(model_path)
model_filename = 'model_059.pth'
autoencoder_name = 'variational convolutional autoencoder'

### Get file path of trained model: 

In [ ]:
models_file_path = data_path / model_name
models = os.listdir(models_file_path)
models.sort(reverse=True)
print(len(models))
models[0:3]

In [ ]:
fully_trained_model = models[0]
model_file_path = str(project_path / model_name / fully_trained_model)
model_file_path


### Check environment variables:

In [ ]:
import os
from pathlib import Path

# Check if .env exists in current directory
env_path = Path(db_env)
print(f"Current directory: {os.getcwd()}")
print(f".env exists: {env_path.exists()}")

# If loaded, check what environment variables are available
print(f"\nDB_NAME: {os.getenv('DB_NAME')}")
print(f"DB_USER: {os.getenv('DB_USER')}")
print(f"DB_HOST: {os.getenv('DB_HOST')}")
print(f"DB_PORT: {os.getenv('DB_PORT')}")

### Get filepaths from label data:

In [ ]:
label_data_from_sources[file_source].head()

In [ ]:

# Get image paths from labels file and convert image paths to Path objects
source_files = [Path(p) for p in label_data_from_sources[file_source]['file_paths']]

print('Number of files:')
print(len(source_files))
print('Compare file id with filepath:')
print(label_data_from_sources[file_source].file_paths[100])
print(label_data_from_sources[file_source].image_id[100])
print(label_data_from_sources[file_source].head())

In [ ]:
source_files[0]

### Get ID's of images in the database (filtered by source):

In [ ]:
# file_source_1:
existing_image_ids = get_existing_image_ids(source_filter=file_source, db_env=db_env, conn=None, cur=None)
print(len(existing_image_ids))
print(existing_image_ids[0:7])

### Initialize MLDataLoader:

In [ ]:
# Initialize MLDataLoader, this also establishes the connection with the database:
loader = MLDataLoader(db_name=database_name, source=None)

### Load images into database:

In [ ]:

print("=" * 70)
print("STEP 1: LOADING IMAGES")
print("=" * 70)

# Extract image information from TIF files
image_ids = []      # Will be integers: [2, 3, 8, 15, ...]
filenames = []      # Will be: ['BernerOberland002.tif', ...]
file_paths = []     # Full paths

for file in source_files:  # tif_files from our earlier exploration
    # Extract numeric ID from filename
    name = file.stem  #
    match = re.search(r'(\d+)$', name)
    
    if match:
        id_int = int(match.group(1))  # '002' → 2 (integer)
        
        image_ids.append(id_int)
        filenames.append(file.name)
        file_paths.append(str(file))

# print(f"Found {len(image_ids)} images")
#print(f"Image IDs (integers): {sorted(image_ids)}")
# print()

# Load images using MLDataLoader
#id_mapping = loader.load_images(
result = loader.load_images_safe(
    image_ids=image_ids,
    filenames=filenames,
    file_paths=file_paths,
    source=file_source
)

print()

id_mapping = result['id_mapping']
new_files = result['inserted_files']
reused_files = result['existing_files']

print(f"✅ ID Mapping created: {len(id_mapping)} entries")
print("Sample mappings (source, source_image_id) → database_image_id:")
for key in sorted(id_mapping.keys())[:5]:
    print(f"  {key} → {id_mapping[key]}")
print('New files:')
print(len(new_files))
print(new_files[0:3])
print('Reused files:')
print(len(reused_files))
print(reused_files[0:3])

### Label data by file source:

In [ ]:
label_data_from_sources = {}

label_data_sources = list(set(label_data.file_source))

for label_data_source in label_data_sources: 
    label_data_from_source = label_data[label_data.file_source==label_data_source]
    
    label_data_from_sources[label_data_source] = label_data_from_source



In [ ]:
print(len(label_data_from_sources))
print(label_data_from_sources.keys())
for source, label_data in label_data_from_sources.items():
    print(source)
    print(label_data.shape)
    print(set(label_data.file_source))
    

In [ ]:
label_data_from_sources[file_source].head()

### Load ground truth into database:

In [ ]:
import pandas as pd

print("=" * 70)
print("STEP 3: LOADING GROUND TRUTH")
print("=" * 70)

file_sources = list(label_data_from_sources.keys())

for file_source_n in file_sources: 
    print(file_source_n)
    label_data_from_source = label_data_from_sources[file_source_n]
    label_data_select = label_data_from_source[['image_id', label_name]]
    
    print(f"Loaded {len(label_data_select)} images")
    
    # Transform label_data from wide to long format
    label_data_long = label_data_select.melt(
        id_vars=['image_id'], 
        var_name='label_name',
        value_name='value'
    )
    
    # Convert image_id to integer (if it's a string)
    label_data_long['image_id'] = label_data_long['image_id'].astype(int)
    
    # Convert value from 0/1 to 'false'/'true'
    label_data_long['value'] = label_data_long['value'].apply(lambda x: 'true' if x == 1 else 'false')
    
    print(f"Transformed {len(label_data)} rows (wide) → {len(label_data_long)} rows (long)")
    print(f"\nFirst few rows:")
    print(label_data_long.head(10))
    print()
    
    # Load ground truth
    result = loader.load_ground_truth_safe(label_data_long, source=file_source_n)
    
    print(f"\n✅ Ground truth loading {file_source_n} complete!")
    print(f"   Inserted: {result['inserted']}")
    print(f"   Existing: {result['existing']}")
    print(f"   Skipped: {result['skipped']}")
    

### Load analysis_runs data: 

In [ ]:
import pickle

#file_path = data_path_comb / metadata_results_file
file_path = data_path / timestamp_id / metadata_results_file
# Load
with open(file_path, 'rb') as f:
    metadata_results_clustering = pickle.load(f)

In [ ]:
metadata_results_clustering.keys()

In [ ]:
metadata_results_clustering['run_timestamp']

In [ ]:
metadata_results_clustering['autoencoder_params']

In [ ]:
metadata_results_clustering['cluster_data'].head()

### Load times data: 

In [ ]:
import pickle
#file_path = data_path_comb / times_file
file_path = data_path / timestamp_id / times_file
# Load
with open(file_path, 'rb') as f:
    times_data = pickle.load(f)

In [ ]:
times_data

### Get numbers of train, validation, and total images processed:

In [ ]:

n_images = metadata_results_clustering['cluster_data'].shape[0]
n_images


In [ ]:
metadata_results_clustering.keys()

In [ ]:
metadata_results_clustering['notes']

In [ ]:
metadata_results_clustering['run_timestamp']
metadata_results_clustering['analysis_type']
metadata_results_clustering['model_name']
metadata_results_clustering['python_script']
times_data['duration_seconds']
metadata_results_clustering['autoencoder_name']
metadata_results_clustering['autoencoder_implementation']
metadata_results_clustering['autoencoder_params']
metadata_results_clustering['dim_reduction_name']
metadata_results_clustering['dim_reduction_implementation']
metadata_results_clustering['dim_reduction_params']
metadata_results_clustering['clustering_name']
metadata_results_clustering['clustering_params']
metadata_results_clustering['clustering_implementation']

In [ ]:
# metadata_results_clustering['python_script'] = 'img_to_pytorch_vae_nv.ipynb'
# metadata_results_clustering['clustering_implementation'] = 'sklearn.cluster'

In [ ]:
metadata_results_clustering['python_script'] = 'img_to_pytorch_vae_nv.ipynb'

In [ ]:
metadata_results_clustering['run_timestamp']
metadata_results_clustering['analysis_type']

### Load analysis runs data into database:

In [ ]:
analysis_run_id = loader.load_analysis_run(run_timestamp = metadata_results_clustering['run_timestamp'], 
                         analysis_type = metadata_results_clustering['analysis_type'], 
                         model_name = metadata_results_clustering['model_name'], 
                         python_script = metadata_results_clustering['python_script'], 
                         model_version=None, 
                         hyperparameters=None, 
                         notes=metadata_results_clustering['notes'], 
                         start_time=None, 
                         duration_seconds = times_data['duration_seconds'][0], 
                         images_processed = n_images, 
                         # Clustering pipeline parameters (all optional) 
                         autoencoder_name = autoencoder_name, 
                         autoencoder_implementation = metadata_results_clustering['autoencoder_implementation'], 
                         autoencoder_file = model_file_path, 
                         autoencoder_params = metadata_results_clustering['autoencoder_params'], 
                         dim_reduction_name = metadata_results_clustering['dim_reduction_name'], 
                         dim_reduction_implementation = metadata_results_clustering['dim_reduction_implementation'], 
                         dim_reduction_params = metadata_results_clustering['dim_reduction_params'], 
                         clustering_name = metadata_results_clustering['clustering_name'], 
                         clustering_implementation = metadata_results_clustering['clustering_implementation'], 
                         clustering_params = metadata_results_clustering['clustering_params'])
analysis_run_id


In [ ]:
#loader.conn.rollback()

In [ ]:
file_sources

### Get cluster data by file source:

In [ ]:
cluster_data = metadata_results_clustering['cluster_data'].copy()
cluster_data.head()

In [ ]:
cluster_data_from_sources = {}

for file_source in file_sources:
    data_source_n = cluster_data[cluster_data.file_source==file_source]
    cluster_data_from_sources[file_source] = data_source_n

In [ ]:
for file_source in list(cluster_data_from_sources.keys()):
    print(file_source)
    print(type(cluster_data_from_sources[file_source]))
    print(cluster_data_from_sources[file_source].shape)
    print(set(cluster_data_from_sources[file_source].file_source))

### Load results into database: 

In [ ]:
for file_source in file_sources:
    print('\n')
    print(file_source)
    cluster_data_from_source = cluster_data_from_sources[file_source]
    print(set(cluster_data_from_source.file_source))
    result = loader.load_clustering_results(
        analysis_run_id=analysis_run_id,
        clustering_dataframe=cluster_data_from_source,
        source=file_source
    )

In [ ]:
# Close cursor and connection
loader.cur.close()
loader.conn.close()

In [ ]:
# ============================================================================
# CLUSTERING DATA VERIFICATION
# ============================================================================
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()

conn = psycopg2.connect(
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT')
)

print("=" * 80)
print("CLUSTERING DATA VERIFICATION")
print("=" * 80)

# ---------------------------------------------------------------------------
# 1. Check Visual Genome Images Loaded
# ---------------------------------------------------------------------------
print("\n1. IMAGES")
print("-" * 80)

query1 = """
SELECT 
    source,
    COUNT(*) as image_count,
    MIN(source_image_id) as min_id,
    MAX(source_image_id) as max_id
FROM images
GROUP BY source;
"""

df1 = pd.read_sql_query(query1, conn)
print(df1.to_string(index=False))

if len(df1) == 0:
    print("\n⚠️  WARNING: No visual_genome images found!")
else:
    print(f"\n✅ Found {df1['image_count'].iloc[0]} Visual Genome images")


In [ ]:
# ---------------------------------------------------------------------------
# 2. Check Clustering Analysis Run
# ---------------------------------------------------------------------------
print("\n\n2. CLUSTERING ANALYSIS RUN DETAILS")
print("-" * 80)

query2 = """
SELECT 
    analysis_run_id,
    run_timestamp,
    model_name,
    analysis_type,
    autoencoder_name,
    autoencoder_implementation,
    dim_reduction_name,
    dim_reduction_implementation,
    clustering_name,
    clustering_implementation
FROM analysis_runs
WHERE analysis_type = 'clustering'
ORDER BY run_timestamp DESC
--LIMIT 1;
"""

df2 = pd.read_sql_query(query2, conn)
df2



In [ ]:
# ============================================================================
# DEFINE CLUSTERING RUN TO VERIFY
# Using the analysis_run_id from the clustering run we just loaded above
# ============================================================================
clustering_run_id = analysis_run_id  # Reuse the run ID from earlier
print(f"Verifying clustering run ID: {clustering_run_id}")

In [ ]:
# ---------------------------------------------------------------------------
# 3. Check Clustering Results Loaded
# ---------------------------------------------------------------------------
print("\n\n3. CLUSTERING RESULTS")
print("-" * 80)

print('clustering_run_id')
print(clustering_run_id)


if clustering_run_id:
    query3 = f"""
    SELECT 
        COUNT(*) as total_assignments,
        COUNT(DISTINCT cluster_id) as num_clusters,
        COUNT(DISTINCT image_id) as images_clustered,
        MIN(cluster_id) as min_cluster,
        MAX(cluster_id) as max_cluster
    FROM clustering_results
    WHERE analysis_run_id = {clustering_run_id};
    """
    
    df3 = pd.read_sql_query(query3, conn)
    print(df3.to_string(index=False))
    
    if df3['total_assignments'].iloc[0] == 0:
        print("\n⚠️  WARNING: No clustering results found!")
    else:
        print(f"\n✅ {df3['total_assignments'].iloc[0]} cluster assignments loaded")
        print(f"   Clusters: {df3['num_clusters'].iloc[0]} unique clusters")
        print(f"   Images: {df3['images_clustered'].iloc[0]} images clustered")
else:
    print("⚠️  Skipping - no clustering run found")

In [ ]:
# ---------------------------------------------------------------------------
# 4. Check Ground Truth (Buildings Label)
# ---------------------------------------------------------------------------
print("\n\n4. GROUND TRUTH - LABEL")
print("-" * 80)

query4 = """
SELECT 
    i.source,
    COUNT(*) as total_labels,
    SUM(CASE WHEN value = 'true' THEN 1 ELSE 0 END) as is_map,
    SUM(CASE WHEN value = 'false' THEN 1 ELSE 0 END) as is_not_map
FROM ground_truth_history gt
JOIN images i ON gt.image_id = i.image_id
WHERE gt.label_name = 'is_map'
AND gt.is_current = TRUE
group by i.source;

--WHERE i.source = 'visual_genome'
  --AND gt.label_name = 'is_map'
  --AND gt.is_current = TRUE;
"""

df4 = pd.read_sql_query(query4, conn)
print(df4.to_string(index=False))

if df4['total_labels'].iloc[0] == 0:
    print("\n⚠️  WARNING: No ground truth found!")
else:
    print(f"\n✅ {df4['total_labels'].iloc[0]} images labeled")

In [ ]:

# ---------------------------------------------------------------------------
# 5. Cluster Distribution
# ---------------------------------------------------------------------------
print("\n\n5. CLUSTER DISTRIBUTION")
print("-" * 80)

if clustering_run_id:
    query5 = f"""
    SELECT 
        cr.cluster_id,
        COUNT(*) as image_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as percentage
    FROM clustering_results cr
    WHERE cr.analysis_run_id = {clustering_run_id}
    GROUP BY cr.cluster_id
    ORDER BY cr.cluster_id;
    """
    
    df5 = pd.read_sql_query(query5, conn)
    print(df5.to_string(index=False))
    print(f"\n✅ Cluster distribution shown above")
else:
    print("⚠️  Skipping - no clustering run found")

In [ ]:

# ---------------------------------------------------------------------------
# 6. Label by Cluster (Combined View)
# ---------------------------------------------------------------------------
print("\n\n6. LABEL DISTRIBUTION BY CLUSTER")
print("-" * 80)

if clustering_run_id:
    query6 = f"""
    SELECT 
        cr.cluster_id,
        COUNT(*) as total_images,
        SUM(CASE WHEN gt.value = 'true' THEN 1 ELSE 0 END) as is_map,
        SUM(CASE WHEN gt.value = 'false' THEN 1 ELSE 0 END) as is_not_map,
        ROUND(
            100.0 * SUM(CASE WHEN gt.value = 'true' THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) as label_percentage
    FROM clustering_results cr
    JOIN images i ON cr.image_id = i.image_id
    LEFT JOIN ground_truth_history gt 
        ON cr.image_id = gt.image_id 
        AND gt.label_name = 'is_map'
        AND gt.is_current = TRUE
    WHERE cr.analysis_run_id = {clustering_run_id}
    GROUP BY cr.cluster_id
    ORDER BY cr.cluster_id;
    """
    
    df6 = pd.read_sql_query(query6, conn)
    print(df6.to_string(index=False))
    print(f"\n✅ Buildings distribution by cluster shown above")
else:
    print("⚠️  Skipping - no clustering run found")

In [ ]:



# ---------------------------------------------------------------------------
# 7. Sample of Clustering Results with Details
# ---------------------------------------------------------------------------
print("\n\n7. SAMPLE CLUSTERING RESULTS (First 10 images)")
print("-" * 80)

if clustering_run_id:
    query7 = f"""
    SELECT 
        i.filename,
        i.source_image_id,
        cr.cluster_id,
        gt.value as is_map
    FROM clustering_results cr
    JOIN images i ON cr.image_id = i.image_id
    LEFT JOIN ground_truth_history gt 
        ON cr.image_id = gt.image_id 
        AND gt.label_name = 'is_map'
        AND gt.is_current = TRUE
    WHERE cr.analysis_run_id = {clustering_run_id}
    ORDER BY i.source_image_id
    LIMIT 10;
    """
    
    df7 = pd.read_sql_query(query7, conn)
    print(df7.to_string(index=False))
else:
    print("⚠️  Skipping - no clustering run found")

# Close connection
conn.close()

print("\n" + "=" * 80)
print("VERIFICATION COMPLETE")
print("=" * 80)